# Concurrent Multi-Agent Workflow with Microsoft Agent Framework

## Overview

This notebook demonstrates a **concurrent (fan-out / fan-in) multi-agent workflow** using the **Microsoft Agent Framework** for Python. Three specialist agents tackle the **same problem** simultaneously from different perspectives, and their results are aggregated into a unified final answer.

| Agent | Perspective |
|-------|-------------|
| **Agent 1 — Technical Advisor** | Evaluates the problem from a software-engineering and architecture standpoint |
| **Agent 2 — Business Strategist** | Analyzes the problem through a business, market-fit, and ROI lens |
| **Agent 3 — Creative Director** | Explores unconventional, user-experience, and brand-narrative approaches |

### Architecture

```
                           User Prompt
                               │
                    ┌──────────┼──────────┐
                    ▼          ▼          ▼
          ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
          │  Agent 1     │ │  Agent 2     │ │  Agent 3     │
          │ Technical    │ │ Business     │ │ Creative     │
          │ Advisor      │ │ Strategist   │ │ Director     │
          └──────┬───────┘ └──────┬───────┘ └──────┬───────┘
                 │                │                │
                 └────────┬───────┘────────────────┘
                          ▼
               ┌─────────────────────┐
               │   Aggregator        │
               │ (Combines Results)  │
               └──────────┬──────────┘
                          ▼
                    Final Output
```

### Key Concepts Demonstrated

- **Concurrent orchestration**: `ConcurrentBuilder` from the Agent Framework — fans out input to all agents, runs them in parallel, and fans in the results
- **Diverse perspectives**: Three independently-operating agents with distinct expertise generate richer problem analysis than any single approach
- **Custom aggregation**: A callback-based aggregator merges all agent outputs into a structured summary
- **Event streaming**: Live event feed from the workflow runtime tracks each agent's lifecycle

### When to Use Concurrent Orchestration

| ✅ Good fit | ❌ Avoid when |
|------------|--------------|
| Tasks can run independently | Agents must build on each other's work |
| Diverse skills improve the outcome | Strict sequential steps are required |
| Speed matters — parallel cuts wait time | Resource quotas limit parallelism |
| Brainstorming, voting, ensemble reasoning | No clear way to resolve contradictions |

## 1. Install Required Dependencies

The following packages power this notebook:

| Package | Purpose |
|---------|---------|
| `agent-framework` | Core agent primitives (`Agent`, `Message`, `WorkflowBuilder`) |
| `agent-framework-azure-ai` | Azure AI provider: `AzureOpenAIResponsesClient` |
| `agent-framework-orchestrations` | `ConcurrentBuilder` for fan-out/fan-in orchestration |
| `azure-identity` | `DefaultAzureCredential` for passwordless auth |
| `python-dotenv` | Load `.env` configuration |

In [29]:
import subprocess, sys

packages = [
    "agent-framework",
    "agent_framework.azure",
    "agent-framework-azure-ai",
    "agent-framework-orchestrations",
    "azure-identity",
    "python-dotenv",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("✅ All packages ready.")
else:
    print("⚠️ Install warnings/errors:")
    print(result.stderr[-2000:])

✅ All packages ready.


## 2. Import Libraries and Configure Environment

We load configuration from the `.env` file in the same directory. The key variable is:

- `AZURE_AI_PROJECT_ENDPOINT` — The Azure AI Foundry project URL  
  (e.g. `https://msf-hacktest.services.ai.azure.com/api/projects/proj-default`)
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` *(optional)* — Model to use (defaults to `gpt-5`)

Authentication uses `AzureCliCredential`, which picks up Azure CLI login credentials automatically.

In [30]:
import os
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from azure.identity.aio import AzureCliCredential

from agent_framework import Agent, Message
from agent_framework.azure import AzureOpenAIResponsesClient
from agent_framework.orchestrations import ConcurrentBuilder

from IPython.display import display, Markdown, HTML

# ── Load .env from the same directory as this notebook ──────────────────────
env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")
load_dotenv(dotenv_path=env_path, override=False)

# ── Configuration ────────────────────────────────────────────────────────────
PROJECT_ENDPOINT = os.environ.get("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-5")

# Shared credential — uses Azure CLI login (ensures correct tenant)
credential = AzureCliCredential()

# ── Validation ───────────────────────────────────────────────────────────────
if not PROJECT_ENDPOINT:
    raise EnvironmentError(
        "AZURE_AI_PROJECT_ENDPOINT is not set.\n"
        "Add it to your .env or set it as an environment variable:\n"
        '  AZURE_AI_PROJECT_ENDPOINT="https://<resource>.services.ai.azure.com/api/projects/<project>"'
    )

print("✅ Configuration loaded")
print(f"   Project endpoint : {PROJECT_ENDPOINT}")
print(f"   Model deployment : {MODEL_DEPLOYMENT}")

✅ Configuration loaded
   Project endpoint : https://admin-1043-resource.services.ai.azure.com/api/projects/admin-1043
   Model deployment : gpt-5.4


## 3. Define Three Specialist Agents

Each agent receives the **same input prompt** but approaches the problem from a distinct perspective:

| Agent | Role | Focus |
|-------|------|-------|
| **Technical Advisor** | Evaluates feasibility, architecture, tech stack | Engineering trade-offs, scalability, implementation complexity |
| **Business Strategist** | Analyzes market opportunity and ROI | Revenue potential, competitive landscape, go-to-market strategy |
| **Creative Director** | Proposes brand narrative and UX innovation | User experience, storytelling, differentiation, design thinking |

All three agents use the same `AzureOpenAIResponsesClient` and model deployment but have specialized instructions.

```
                 Same Input Prompt
                 ┌──────┼──────┐
                 ▼      ▼      ▼
           Technical Business Creative
            Advisor  Strategist Director
```

In [31]:
# ── Shared chat client ────────────────────────────────────────────────────────
local_client = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=MODEL_DEPLOYMENT,
    credential=credential,
)

# ═══════════════════════════════════════════════════════════════════════════════
# Agent 1 — Technical Advisor
# ═══════════════════════════════════════════════════════════════════════════════
TECHNICAL_INSTRUCTIONS = """\
You are a senior software architect and technical advisor. When given a product
idea or business challenge, you evaluate it from a **technical feasibility**
perspective.

Your analysis must cover:
1. **Architecture** — High-level system design and key components
2. **Tech Stack** — Recommended languages, frameworks, and services
3. **Scalability** — How the solution handles growth (users, data, throughput)
4. **Risks** — Technical debt, integration complexity, security concerns
5. **Effort Estimate** — Rough T-shirt sizing (S / M / L / XL) with justification

Rules:
- Be specific — name actual technologies, not generic categories.
- Keep the response under 300 words.
- Use numbered sections matching the five areas above.
- End with a one-line **Technical Verdict**: "Feasible / Feasible with caveats / Not recommended".
"""

agent_technical = Agent(
    client=local_client,
    name="TechnicalAdvisor",
    instructions=TECHNICAL_INSTRUCTIONS,
    description="Evaluates ideas from a software architecture and engineering perspective.",
)

# ═══════════════════════════════════════════════════════════════════════════════
# Agent 2 — Business Strategist
# ═══════════════════════════════════════════════════════════════════════════════
BUSINESS_INSTRUCTIONS = """\
You are a seasoned business strategist and market analyst. When given a product
idea or initiative, you evaluate it from a **business and market-fit** perspective.

Your analysis must cover:
1. **Market Opportunity** — Target audience size, demand signals, and trends
2. **Competitive Landscape** — Key competitors and differentiation potential
3. **Revenue Model** — Monetization strategy (subscription, usage-based, etc.)
4. **Go-to-Market** — Launch strategy and key channels
5. **ROI Projection** — Rough payback period and expected value

Rules:
- Ground your analysis in realistic assumptions.
- Keep the response under 300 words.
- Use numbered sections matching the five areas above.
- End with a one-line **Business Verdict**: "Strong opportunity / Moderate opportunity / Weak opportunity".
"""

agent_business = Agent(
    client=local_client,
    name="BusinessStrategist",
    instructions=BUSINESS_INSTRUCTIONS,
    description="Evaluates ideas from a business strategy and market analysis perspective.",
)

# ═══════════════════════════════════════════════════════════════════════════════
# Agent 3 — Creative Director
# ═══════════════════════════════════════════════════════════════════════════════
CREATIVE_INSTRUCTIONS = """\
You are an innovative creative director and UX visionary. When given a product
idea or challenge, you evaluate it from a **user experience and brand narrative**
perspective.

Your analysis must cover:
1. **User Story** — The core user pain point and the emotional journey
2. **Brand Narrative** — How this product tells a compelling story
3. **UX Innovation** — Novel interaction patterns or design ideas
4. **Differentiation** — What makes this stand out visually and experientially
5. **Viral Potential** — Elements that encourage sharing and word-of-mouth

Rules:
- Be bold and imaginative — push beyond conventional thinking.
- Keep the response under 300 words.
- Use numbered sections matching the five areas above.
- End with a one-line **Creative Verdict**: "Exciting / Promising / Needs more spark".
"""

agent_creative = Agent(
    client=local_client,
    name="CreativeDirector",
    instructions=CREATIVE_INSTRUCTIONS,
    description="Evaluates ideas from a user experience and brand narrative perspective.",
)

print("✅ All three agents ready")
print(f"   Agent 1: {agent_technical.name} (Technical)")
print(f"   Agent 2: {agent_business.name} (Business)")
print(f"   Agent 3: {agent_creative.name} (Creative)")
print(f"   Model  : {MODEL_DEPLOYMENT}")

✅ All three agents ready
   Agent 1: TechnicalAdvisor (Technical)
   Agent 2: BusinessStrategist (Business)
   Agent 3: CreativeDirector (Creative)
   Model  : gpt-5.4


## 4. Build the Concurrent Workflow

The `ConcurrentBuilder` wires a **fan-out / fan-in** pattern:

1. **Dispatcher** (internal) — broadcasts the user prompt to all agents simultaneously
2. **Fan-out** — each agent runs in parallel, independently processing the same input
3. **Fan-in** — all `AgentExecutorResponse` objects are collected once every agent finishes
4. **Aggregator** — combines the individual responses into the final output

We use the **default aggregator**, which returns a `list[ChatMessage]` containing one user prompt followed by each agent's final assistant message.

```
Input ──► Dispatcher ──┬──► TechnicalAdvisor  ──┐
                       ├──► BusinessStrategist ──┤──► Aggregator ──► Output
                       └──► CreativeDirector   ──┘
```

> **Note:** Agents operate independently and do **not** share results with each other. Each agent sees only the original user prompt.

In [32]:
# ── Build the concurrent workflow ─────────────────────────────────────────────
workflow = ConcurrentBuilder(
    participants=[agent_technical, agent_business, agent_creative],
).build()

print("✅ Concurrent workflow built")
print(f"   Participants : {agent_technical.name}, {agent_business.name}, {agent_creative.name}")
print(f"   Entry types  : {[t.__name__ for t in workflow.input_types]}")
print(f"   Output types : {[t.__name__ for t in workflow.output_types]}")

✅ Concurrent workflow built
   Participants : TechnicalAdvisor, BusinessStrategist, CreativeDirector
   Entry types  : ['Message', 'list', 'AgentExecutorRequest', 'str']
   Output types : ['AgentResponseUpdate', 'AgentResponse', 'list']


## 5. Execute the Workflow

The cell below:
1. Sends a product-idea prompt to the concurrent workflow using **streaming mode** (`stream=True`)
2. Captures lifecycle events (started, executor invoked/completed, data, failed)
3. Extracts the aggregated output containing each agent's perspective
4. Renders each agent's analysis as rich Markdown

All three agents process the same prompt **simultaneously** — their responses arrive as each finishes, and the final output is available once all are complete.

In [33]:
# ── Product idea prompt to evaluate concurrently ─────────────────────────────
USER_PROMPT = (
    "We want to build an AI-powered personal finance assistant that helps users "
    "track spending, set savings goals, and receive personalized investment "
    "recommendations. The target market is millennials and Gen Z in Europe. "
    "Evaluate this product idea from your area of expertise."
)

display(Markdown(f"## 🚀 Concurrent Workflow\n\n**Prompt:** *{USER_PROMPT}*\n\n---"))

# ── Run with streaming ───────────────────────────────────────────────────────
stream = workflow.run(USER_PROMPT, stream=True)

# Track lifecycle events
completed_executors = []
agent_outputs = {}

async for event in stream:
    etype = event.type
    ts = datetime.now().strftime("%H:%M:%S")

    if etype == "started":
        print(f"[{ts}] 🟢 Workflow started — all agents running in parallel")

    elif etype == "executor_invoked":
        executor_id = getattr(event, "executor_id", "?")
        print(f"[{ts}] 🚀 Running: {executor_id}")

    elif etype == "executor_completed":
        executor_id = getattr(event, "executor_id", "?")
        completed_executors.append(executor_id)
        print(f"[{ts}] ✅ Completed: {executor_id}")

    elif etype == "data":
        executor_id = getattr(event, "executor_id", "?")
        if isinstance(event.data, list):
            agent_outputs[executor_id] = event.data

    elif etype == "failed" or etype == "executor_failed":
        details = getattr(event, "details", event.data)
        print(f"[{ts}] ❌ Failed: {details}")

print(f"\n✅ Workflow complete — {len(completed_executors)} executors ran")

# ── Extract final result ─────────────────────────────────────────────────────
result = await stream.get_final_response()
outputs = result.get_outputs()

# ── Display each agent's response ────────────────────────────────────────────
if outputs:
    final_messages = outputs[-1] if isinstance(outputs[-1], list) else outputs

    # The default aggregator returns [user_prompt, agent1_reply, agent2_reply, agent3_reply]
    agent_names = ["🔧 Technical Advisor", "💼 Business Strategist", "🎨 Creative Director"]
    agent_idx = 0

    for msg in final_messages:
        role = msg.role if hasattr(msg, "role") else ""
        text = msg.text if hasattr(msg, "text") else str(msg)

        if str(role).lower() == "assistant" and text:
            label = agent_names[agent_idx] if agent_idx < len(agent_names) else f"Agent {agent_idx + 1}"
            display(Markdown(f"---\n## {label}\n"))
            display(Markdown(text))
            agent_idx += 1

if agent_idx == 0:
    print("⚠️ No agent outputs captured.")
    if outputs:
        for output in outputs:
            if isinstance(output, list):
                for msg in output:
                    text = msg.text if hasattr(msg, "text") else str(msg)
                    if text:
                        print(text)

display(Markdown("---\n*All agents completed. Workflow finished.*"))

## 🚀 Concurrent Workflow

**Prompt:** *We want to build an AI-powered personal finance assistant that helps users track spending, set savings goals, and receive personalized investment recommendations. The target market is millennials and Gen Z in Europe. Evaluate this product idea from your area of expertise.*

---

[12:00:00] 🟢 Workflow started — all agents running in parallel
[12:00:00] 🚀 Running: dispatcher
[12:00:00] ✅ Completed: dispatcher
[12:00:00] 🚀 Running: TechnicalAdvisor
[12:00:00] 🚀 Running: BusinessStrategist
[12:00:00] 🚀 Running: CreativeDirector
[12:00:16] ✅ Completed: BusinessStrategist
[12:00:17] ✅ Completed: CreativeDirector
[12:00:17] ✅ Completed: TechnicalAdvisor
[12:00:17] 🚀 Running: aggregator
[12:00:17] ✅ Completed: aggregator

✅ Workflow complete — 5 executors ran


---
## 🔧 Technical Advisor


1. **Architecture** — Mobile-first app (React Native) backed by a modular backend. Core services: **auth/user profile**, **bank aggregation**, **transaction categorization**, **goal tracking**, **recommendation engine**, and **notifications**. Use event-driven ingestion for bank transactions via open banking APIs, store normalized financial data, and expose features through a GraphQL or REST API. AI layer should separate **LLM-based assistant/chat** from **rules + ML models** for budgeting and recommendations. Add an admin/compliance console for audit logs, consent management, and model review.

2. **Tech Stack** — Frontend: **React Native** + TypeScript. Backend: **Python FastAPI** for AI/data services and **Node.js/NestJS** for user-facing APIs if needed. Data: **PostgreSQL** for transactional data, **Redis** for caching, **S3-compatible object storage** for documents/logs. AI: **OpenAI or Anthropic API** for conversational UX, **scikit-learn/XGBoost** for categorization and risk scoring. Integrations: **Tink, TrueLayer, or Plaid Europe** for PSD2/open banking. Infra: **AWS** (ECS/EKS, RDS, CloudWatch), **Terraform**, **Auth0** or **AWS Cognito**.

3. **Scalability** — Start as a modular monolith, then split high-load services (transaction ingestion, recommendations, notifications). Use queues (**SQS/Kafka**) for bank sync and model pipelines. PostgreSQL can scale well initially with read replicas and partitioning by user/account. Cache balances and dashboards in Redis. AI inference costs can be controlled with async processing, retrieval, and prompt caching.

4. **Risks** — Biggest issues are **regulatory**: GDPR, PSD2, consent flows, and if “investment recommendations” cross into **MiFID II regulated advice**. Security requirements are high: encryption, auditability, least privilege, anomaly detection. Bank API variability and data quality will create integration overhead. LLM hallucinations make ungoverned financial advice risky; require guardrails and human-reviewed recommendation logic.

5. **Effort Estimate** — **XL**. A basic budgeting assistant is feasible in 6–9 months, but compliant investment recommendations across Europe significantly increase legal, data, and security complexity.

**Technical Verdict:** Feasible with caveats

---
## 💼 Business Strategist


1. **Market Opportunity** — Large and growing. Europe has tens of millions of millennials and Gen Z consumers, with strong adoption of mobile banking, budgeting apps, and neo-brokers. Demand signals are favorable: inflation, cost-of-living pressure, and rising retail investing interest all increase need for smarter financial guidance. However, willingness to pay is uneven; budgeting tools are common, but users pay more readily for automation, tax optimization, and investment value.

2. **Competitive Landscape** — Crowded. Competitors include budgeting apps (Emma, YNAB, Plum), digital banks (Revolut, N26, Monzo in some markets), and robo-advisors/investing apps (Scalable Capital, Trade Republic). Differentiation will be difficult unless the product combines: unified account aggregation, highly personalized AI coaching, goal-based saving automation, and compliant, transparent investment recommendations. Trust, UX, and regulatory credibility are critical moats.

3. **Revenue Model** — Best model is freemium + premium subscription (€5–15/month) for advanced insights, investment guidance, and automation. Secondary revenue could include referral/affiliate fees from savings, insurance, or broker partners, and potentially AUM-style fees if managing investments directly. Pure ad-based monetization would hurt trust.

4. **Go-to-Market** — Start in 1–2 countries with strong open-banking infrastructure and digital-finance adoption (e.g., UK if included, Germany, Netherlands, France). Launch with budgeting/savings first, then layer investment recommendations after establishing trust. Channels: TikTok/Instagram creators in personal finance, app-store ASO, partnerships with employers/benefits platforms, and referral incentives. Strong compliance messaging and social proof will matter.

5. **ROI Projection** — Moderate but attractive if CAC is controlled. Assume CAC of €20–50 via digital and referral channels, premium conversion of 3–8%, and annual ARPU of €60–120 for paid users. Payback could take 12–24 months unless viral/referral loops reduce CAC. Regulatory/compliance costs may delay profitability but increase defensibility.

**Business Verdict: Moderate opportunity**

---
## 🎨 Creative Director


1. **User Story** — Millennials and Gen Z in Europe don’t just struggle with budgeting; they feel punished by finance. Banking apps show numbers, but not meaning. The pain point is emotional fragmentation: “Am I doing okay?” “Can I still enjoy life and be responsible?” The ideal journey is from guilt and confusion to calm control—an assistant that feels like a money coach, not an accountant.

2. **Brand Narrative** — This product should position itself as the “financial co-pilot for modern life,” translating intimidating money decisions into confident micro-actions. The story isn’t wealth flexing—it’s freedom, clarity, and future-proofing. For a generation facing inflation, housing pressure, and unstable career paths, the brand can become a symbol of self-trust: smart with money without becoming obsessed by it.

3. **UX Innovation** — Go beyond dashboards. Imagine a conversational timeline that explains spending like a story: “You spent more on going out this month because of travel weekends.” Add mood-based goal setting (“save for peace of mind,” “save for adventure”), AI-generated weekly “money rituals,” and a visual “future self” simulator showing trade-offs between spending today and investing tomorrow. Make investing recommendations feel native, educational, and low-pressure.

4. **Differentiation** — Most fintech products feel sterile, masculine, and number-heavy. This can stand out through emotionally intelligent UX, warmer language, and a visually editorial interface—part coach, part lifestyle companion. Hyper-localization for Europe is key: currencies, regulations, tax wrappers, and cultural spending habits should feel seamlessly understood.

5. **Viral Potential** — Shareable “money wins” are powerful if framed with taste: streaks, savings milestones, “AI spotted €240 you could redirect,” or personalized financial archetypes. Social-friendly annual recaps (“how your money story evolved this year”) could become this product’s version of Spotify Wrapped.

**Creative Verdict:** Exciting

---
*All agents completed. Workflow finished.*

## 6. Custom Aggregator — Executive Summary

The default aggregator simply collects each agent's response. But what if you want a **synthesized summary** that merges insights from all perspectives?

The `ConcurrentBuilder` supports a custom aggregator via `.with_aggregator()`. You can pass either:
- An `Executor` subclass with a `@handler` method
- A **callback function** that receives `list[AgentExecutorResponse]` and returns a result

Below, we rebuild the workflow with a callback-based aggregator that creates a structured executive summary combining all three perspectives.

In [34]:
from agent_framework._workflows._agent_executor import AgentExecutorResponse

# ── Custom aggregator callback ────────────────────────────────────────────────
def executive_summary(results: list[AgentExecutorResponse]) -> str:
    """Combine all agent responses into a structured executive summary."""
    sections = []
    perspective_labels = ["🔧 Technical Perspective", "💼 Business Perspective", "🎨 Creative Perspective"]

    for i, r in enumerate(results):
        # Extract the final assistant message from each agent's response
        resp_messages = list(getattr(r.agent_response, "messages", []) or [])
        assistant_text = None
        for msg in reversed(resp_messages):
            role = getattr(msg, "role", "")
            if str(role).lower() == "assistant":
                assistant_text = msg.text if hasattr(msg, "text") else str(msg)
                break

        label = perspective_labels[i] if i < len(perspective_labels) else f"Perspective {i + 1}"
        if assistant_text:
            sections.append(f"### {label}\n\n{assistant_text}")
        else:
            sections.append(f"### {label}\n\n*No response received.*")

    header = "# 📋 Executive Summary — Multi-Perspective Analysis\n\n"
    header += "The following analysis was generated by three independent AI agents "
    header += "working **concurrently** on the same prompt. Each agent applied its "
    header += "specialized expertise to evaluate the product idea.\n"

    footer = "\n---\n### 🏁 Combined Assessment\n\n"
    footer += f"**Perspectives gathered:** {len(results)}\n\n"
    footer += "Review the verdicts from each perspective above to form a holistic decision."

    return header + "\n\n---\n\n".join(sections) + footer


# ── Build workflow with custom aggregator ────────────────────────────────────
workflow_custom = ConcurrentBuilder(
    participants=[agent_technical, agent_business, agent_creative],
).with_aggregator(executive_summary).build()

print("✅ Concurrent workflow with custom aggregator built")
print(f"   Aggregator: executive_summary (callback-based)")

✅ Concurrent workflow with custom aggregator built
   Aggregator: executive_summary (callback-based)


## 7. Execute with Custom Aggregator

Now we run the same product idea through the concurrent workflow, but this time the custom aggregator combines all perspectives into a single executive summary document.

In [35]:
# ── Different prompt for the second run ───────────────────────────────────────
USER_PROMPT_2 = (
    "We are considering launching a subscription-based online platform that uses "
    "AI to generate personalized meal plans and grocery lists based on dietary "
    "preferences, health goals, and local store availability. The initial target "
    "market is health-conscious professionals aged 25–45 in North America. "
    "Evaluate this product idea from your area of expertise."
)

display(Markdown(f"## 🚀 Concurrent Workflow — Custom Aggregator\n\n**Prompt:** *{USER_PROMPT_2}*\n\n---"))

# ── Run with streaming ───────────────────────────────────────────────────────
stream2 = workflow_custom.run(USER_PROMPT_2, stream=True)

completed_executors_2 = []
async for event in stream2:
    etype = event.type
    ts = datetime.now().strftime("%H:%M:%S")

    if etype == "started":
        print(f"[{ts}] 🟢 Workflow started — all agents running in parallel")
    elif etype == "executor_invoked":
        executor_id = getattr(event, "executor_id", "?")
        print(f"[{ts}] 🚀 Running: {executor_id}")
    elif etype == "executor_completed":
        executor_id = getattr(event, "executor_id", "?")
        completed_executors_2.append(executor_id)
        print(f"[{ts}] ✅ Completed: {executor_id}")
    elif etype == "failed" or etype == "executor_failed":
        details = getattr(event, "details", event.data)
        print(f"[{ts}] ❌ Failed: {details}")

print(f"\n✅ Workflow complete — {len(completed_executors_2)} executors ran")

# ── Get the aggregated result ────────────────────────────────────────────────
result2 = await stream2.get_final_response()
outputs2 = result2.get_outputs()

# The custom aggregator returns a string (the executive summary)
if outputs2:
    summary = outputs2[-1] if isinstance(outputs2[-1], str) else str(outputs2[-1])
    display(Markdown(summary))
else:
    print("⚠️ No outputs received from the workflow.")

display(Markdown("---\n*Custom aggregator workflow complete.*"))

## 🚀 Concurrent Workflow — Custom Aggregator

**Prompt:** *We are considering launching a subscription-based online platform that uses AI to generate personalized meal plans and grocery lists based on dietary preferences, health goals, and local store availability. The initial target market is health-conscious professionals aged 25–45 in North America. Evaluate this product idea from your area of expertise.*

---

[12:00:17] 🟢 Workflow started — all agents running in parallel
[12:00:17] 🚀 Running: dispatcher
[12:00:17] ✅ Completed: dispatcher
[12:00:17] 🚀 Running: TechnicalAdvisor
[12:00:17] 🚀 Running: BusinessStrategist
[12:00:17] 🚀 Running: CreativeDirector
[12:00:28] ✅ Completed: TechnicalAdvisor
[12:00:29] ✅ Completed: BusinessStrategist
[12:00:36] ✅ Completed: CreativeDirector
[12:00:36] 🚀 Running: executive_summary
[12:00:36] ✅ Completed: executive_summary

✅ Workflow complete — 5 executors ran


# 📋 Executive Summary — Multi-Perspective Analysis

The following analysis was generated by three independent AI agents working **concurrently** on the same prompt. Each agent applied its specialized expertise to evaluate the product idea.
### 🔧 Technical Perspective

1. **Architecture** — Web + mobile-friendly frontend (Next.js) backed by a modular API layer. Core services: user/profile service, subscription/billing, meal-plan generation, grocery-list optimization, store-inventory integration, and analytics. Use an orchestration layer to combine user constraints (allergies, macros, goals), recipe database, and LLM outputs. Persist structured nutrition/recipe data separately from AI prompts/results. Add admin tooling for recipe curation and moderation.

2. **Tech Stack** — Frontend: Next.js + TypeScript. Backend: Python with FastAPI for AI/nutrition logic, plus Node.js if needed for webhooks/billing. Database: PostgreSQL for users/subscriptions/recipes; Redis for caching session and plan results. Search/filtering: OpenSearch or PostgreSQL full-text initially. AI: OpenAI or Anthropic APIs for personalization text generation, paired with deterministic rules engine for dietary safety. Integrations: Stripe for subscriptions, Nutritionix/USDA FoodData Central for nutrition, Instacart/Walmart/Kroger APIs where available for store data. Host on AWS (ECS/Fargate, RDS, ElastiCache, S3).

3. **Scalability** — Generate plans asynchronously via job queue (Celery/SQS) to handle peak demand and reduce API latency. Cache common recipe components and store normalized ingredient mappings. Keep AI calls bounded with templates and retrieval from curated recipes to control cost. Multi-tenant-ready architecture is not required initially, but API-first design supports future B2B partnerships.

4. **Risks** — Store availability APIs are fragmented and region-dependent; “local availability” may be incomplete. LLM hallucinations can create unsafe recommendations for allergies/medical diets, so hard validation rules are mandatory. Nutrition/licensing data quality, HIPAA-adjacent expectations, and privacy compliance (CCPA/PIPEDA) need attention. AI cost can erode margins if generation is too frequent.

5. **Effort Estimate** — **L**. Core MVP is feasible in 4–6 months with 5–8 engineers if you narrow scope: curated recipe set, limited store integrations, and AI-assisted personalization rather than fully autonomous meal design.

**Technical Verdict:** Feasible with caveats

---

### 💼 Business Perspective

1. **Market Opportunity** — Attractive niche with meaningful demand. In North America, the target segment of health-conscious professionals 25–45 is sizable and growing, driven by higher spending on wellness, time-saving tools, and personalized nutrition. Demand signals include growth in meal-kit adoption, nutrition apps, wearable integration, and interest in GLP-1-friendly, high-protein, allergy-aware, and goal-based eating. Pain point is real: people want healthier eating without planning overhead. Market size is large enough for a focused premium SaaS entry, though broad consumer nutrition is crowded.

2. **Competitive Landscape** — Competitors include meal-planning apps (Eat This Much, PlateJoy, eMeals), grocery platforms (Instacart, Walmart, Kroger), and health apps (MyFitnessPal, Noom). Most offer partial personalization, but fewer combine AI meal planning + grocery list automation + local store availability in one workflow. Differentiation potential is strongest if recommendations are highly accurate, save money/time, and integrate directly into shopping behavior. Weak differentiation if it becomes “another recipe generator.”

3. **Revenue Model** — Best fit is freemium to subscription: free basic plans, premium at roughly $10–20/month or $80–150/year. Upside from affiliate revenue on grocery orders, sponsored product placement, and B2B2C partnerships with employers, insurers, or wellness programs. Subscription alone can work if retention is strong; grocery-linked commissions improve LTV.

4. **Go-to-Market** — Launch via Instagram/TikTok creators in fitness, nutrition, and busy-professional lifestyle niches; pair with paid Meta/Google acquisition around keywords like “meal plan for macros” or “healthy grocery list app.” Early traction can come from dietitians, gyms, and corporate wellness pilots. Focus MVP on 2–3 use cases: fat loss, muscle gain, and family healthy eating.

5. **ROI Projection** — Moderate upfront build and data-integration cost. If CAC lands around $40–80 and annual gross profit per retained user reaches $80–150, payback could occur in 6–12 months with decent retention. Risk is churn after novelty fades; value must be habitual and shopping-linked.

**Business Verdict: Moderate opportunity**

---

### 🎨 Creative Perspective

1. **User Story** — This audience isn’t just trying to “eat healthy”; they’re fighting decision fatigue, time scarcity, and guilt. The real pain: knowing what they *should* eat but lacking the mental bandwidth to plan, shop, and stay consistent. The emotional arc should move from overwhelm → relief → confidence → identity: “I’m someone who effortlessly eats well.”

2. **Brand Narrative** — Position the platform as a *personal nutrition co-pilot*, not a meal-planning tool. The story is empowerment through intelligent simplification: AI that understands your body, calendar, tastes, and budget better over time. The brand should feel calm, credible, and aspirational—less “diet app,” more “lifestyle command center for modern wellness.”

3. **UX Innovation** — The breakthrough isn’t just personalization; it’s adaptive orchestration. Imagine a “week in one tap” flow: meals, grocery list, substitutions, prep timing, and store-aware optimization generated instantly. Add dynamic modes like “Busy Week,” “High-Protein Reset,” or “Low-Effort Comfort.” A conversational interface that explains *why* each meal fits the user’s goals would build trust and habit stickiness.

4. **Differentiation** — Most competitors stop at recipes. Your edge is *contextual intelligence*: integrating dietary goals with local inventory, pricing, and real-life constraints. Visually, avoid generic wellness clichés. Build a premium system with elegant dashboards, frictionless swaps, and satisfying progress signals—something that feels as polished as a fintech app, but warmer and more human.

5. **Viral Potential** — Sharing shouldn’t be an afterthought. Create socially native outputs: beautiful weekly meal snapshots, “AI planned my healthy week for $87” moments, partner/couple plan syncing, and office-friendly lunch bragability. People share systems that make them look disciplined, efficient, and in-the-know.

**Creative Verdict:** **Promising**
---
### 🏁 Combined Assessment

**Perspectives gathered:** 3

Review the verdicts from each perspective above to form a holistic decision.

---
*Custom aggregator workflow complete.*

## 8. Cleanup

Close the credential to release resources. This cell is optional — resources are also released when the kernel shuts down.

In [36]:
await credential.close()
print("✅ Resources released.")

✅ Resources released.
